In [ ]:
!pip -q install --upgrade pandas scikit-learn tqdm pillow matplotlib --progress-bar off
import os, json, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models as tvm
from tqdm.auto import tqdm
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
USE_DRIVE = True
DRIVE_DATA_DIR = '/content/drive/MyDrive/new-plant-diseases' 
OUT_DIR = '/content/drive/MyDrive/plant_disease_runs_imagefolder'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(OUT_DIR, exist_ok=True)
TRAIN_DIR = f'{DRIVE_DATA_DIR}/train'
VAL_DIR   = f'{DRIVE_DATA_DIR}/valid'
print('Train exists:', os.path.isdir(TRAIN_DIR))
print('Valid exists:', os.path.isdir(VAL_DIR))
print('OUT_DIR:', OUT_DIR)

In [ ]:
USE_LOCAL_COPY = False
if USE_LOCAL_COPY:
    !mkdir -p /content/data/newplant
    !rsync -ah --info=progress2 "$DRIVE_DATA_DIR/train/" "/content/data/newplant/train/"
    !rsync -ah --info=progress2 "$DRIVE_DATA_DIR/valid/" "/content/data/newplant/valid/"
    TRAIN_DIR = '/content/data/newplant/train'
    VAL_DIR   = '/content/data/newplant/valid'
    print('Copied to /content.')

In [ ]:
IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]
IMG_SIZE = 224
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=val_tfms)
num_classes = len(train_ds.classes)
print('Classes:', num_classes)
label_map = {cls:i for i, cls in enumerate(train_ds.classes)}
with open(os.path.join(OUT_DIR, 'label_map.json'), 'w', encoding='utf-8') as f:
    json.dump(label_map, f, indent=2)
print('Saved label_map.json to OUT_DIR')

In [ ]:
BALANCE_MODE = 'none' 
from collections import Counter
train_targets = [y for _, y in train_ds.samples]
counts = Counter(train_targets)
class_weights_tensor = None; sampler=None
if BALANCE_MODE=='class_weights':
    total = sum(counts.values()); import numpy as np, torch
    w = np.array([total/max(1, counts.get(c,1)) for c in range(num_classes)], dtype='float32')
    w = w / w.mean(); class_weights_tensor = torch.tensor(w)
elif BALANCE_MODE=='weighted_sampler':
    from torch.utils.data import WeightedRandomSampler
    cls_w = {c: 1.0/counts[c] for c in counts}
    sample_w = [cls_w[y] for y in train_targets]
    sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_w), num_samples=len(sample_w), replacement=True)

In [ ]:
BATCH_SIZE = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pinmem = (device=='cuda'); num_workers = 2 if device=='cuda' else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=(sampler is None), sampler=sampler, num_workers=num_workers, pin_memory=pinmem)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=pinmem)
len(train_loader), len(val_loader)

In [ ]:
MODEL_NAME='efficientnet_b0'  
PRETRAINED=True
def build_model(name, num_classes, pretrained=True):
    name=name.lower()
    if name=='resnet50':
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m
    elif name=='efficientnet_b0':
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes); return m
    elif name=='vit_b_16':
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes); return m
    else:
        raise ValueError('unknown model')
model = build_model(MODEL_NAME, num_classes, PRETRAINED).to(device)
model

In [ ]:
LR=3e-4; WEIGHT_DECAY=1e-4; LABEL_SMOOTHING=0.0; OPTIMIZER='adam'; MIXED_PRECISION=True
criterion = nn.CrossEntropyLoss(weight=(class_weights_tensor.to(device) if class_weights_tensor is not None else None), label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY) if OPTIMIZER=='adam' else torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
from contextlib import nullcontext
try:
    from torch.amp import GradScaler, autocast
    scaler = GradScaler('cuda') if (MIXED_PRECISION and device=='cuda') else None
except Exception:
    scaler=None; autocast = lambda *a, **k: nullcontext()

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, grad_clip=1.0):
    model.train(); total=0.0
    for x,y in tqdm(loader, leave=False):
        x=x.to(device); y=y.to(device)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast('cuda'):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(x); loss = criterion(logits, y)
            loss.backward();
            if grad_clip is not None: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        total += loss.item()*x.size(0)
    return total/len(loader.dataset)
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); total=0.0; yt=[]; yp=[]
    for x,y in tqdm(loader, leave=False):
        x=x.to(device); y=y.to(device)
        logits = model(x); loss = criterion(logits, y)
        total += loss.item()*x.size(0)
        pred = logits.softmax(1).argmax(1)
        yt.append(y.cpu().numpy()); yp.append(pred.cpu().numpy())
    y_true=np.concatenate(yt); y_pred=np.concatenate(yp)
    acc=float(accuracy_score(y_true, y_pred)); f1M=float(f1_score(y_true,y_pred,average='macro')); f1m=float(f1_score(y_true,y_pred,average='micro'))
    return total/len(loader.dataset), {'accuracy':acc,'f1_macro':f1M,'f1_micro':f1m}

In [ ]:
EPOCHS=12; EARLY_STOP=5; FREEZE_EPOCHS=1
best_acc=-1.0; no_improve=0
best_path=os.path.join(OUT_DIR,'best_model.pth')
if FREEZE_EPOCHS>0:
    for n,p in model.named_parameters():
        if ('fc' in n) or ('classifier.1' in n) or ('heads.head' in n): p.requires_grad=True
        else: p.requires_grad=False
for epoch in range(1,EPOCHS+1):
    if epoch==FREEZE_EPOCHS+1:
        for p in model.parameters(): p.requires_grad=True
    tr=train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    vl,met=evaluate(model, val_loader, criterion, device)
    acc=met['accuracy']
    print(f"Epoch {epoch:02d} | train={tr:.4f} val={vl:.4f} acc={acc:.4f} f1M={met['f1_macro']:.4f} f1m={met['f1_micro']:.4f}")
    if acc>best_acc:
        best_acc=acc; no_improve=0
        torch.save({'epoch':epoch,'model_state':model.state_dict(),'metrics':met,'classes':train_ds.classes}, best_path)
    else:
        no_improve+=1
        if no_improve>=EARLY_STOP:
            print('Early stopping.'); break
print('Best acc:', best_acc, '| saved:', best_path)

In [ ]:
ckpt = torch.load(best_path, map_location=device)
def build_model(name, num_classes, pretrained=True):
    name=name.lower()
    if name=='resnet50':
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m
    elif name=='efficientnet_b0':
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes); return m
    elif name=='vit_b_16':
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
        m.heads.head = nn.Linear(m.heads.head.in_features, num_classes); return m
    else:
        raise ValueError('unknown model')
eval_model = build_model(MODEL_NAME, num_classes, PRETRAINED).to(device)
eval_model.load_state_dict(ckpt['model_state'], strict=False)
vl, met = evaluate(eval_model, val_loader, criterion, device)
print(json.dumps(met, indent=2))